# Sistema de recomendación de cursos en plataforma e-learning: demo de tracking en MLflow

**Caso ficticio.** Plataforma e-learning de la Fundación Ayudinga (Panamá). Pregunta de negocio: *¿qué curso deberíamos recomendar a cada estudiante?* Todos los datos son **sintéticos**.

**Qué hace este notebook.** Entrena y compara 3 modelos y registra todo en MLflow:

| Elemento | Valor |
|---|---|
| Experimento | `course-recommender-demo` |
| Run padre | `model-comparison` |
| Runs hijos | `popularity`, `logreg`, `hgb` |
| Métrica primaria | `ndcg_at_5` |
| Run adicional | `reproducibility-check-<modelo>` (sección 7) |

**Cómo correrlo.** Define el URI del servidor de una de estas dos formas y luego usa *Restart & Run All*:

- Archivo `.env` junto al notebook (no se versiona) con la línea `MLFLOW_TRACKING_URI=http://<servidor-del-DE>:5000`. Para probar en local: `MLFLOW_TRACKING_URI=sqlite:///mlflow.db`.
- Variable de entorno `MLFLOW_TRACKING_URI`, definida antes de abrir Jupyter. Si existe, tiene prioridad sobre el `.env`.

Si el servidor exige autenticación, usa solo variables de entorno (`MLFLOW_TRACKING_USERNAME`, `MLFLOW_TRACKING_PASSWORD` o `MLFLOW_TRACKING_TOKEN`); nunca credenciales en el notebook.

## 0. Configuración

Importa las librerías y define las constantes que controlan todo el notebook: el URI del servidor (leído de una variable de entorno), el nombre del experimento, una única semilla `SEED` y el tamaño de la lista recomendada `K`. Si el URI no está definido, se usa un texto falso a propósito, que la sección 1 detecta para detenerse.

Si hay un archivo `.env` en la carpeta del notebook, sus variables se cargan aquí, sin pisar las que ya estén definidas en el entorno.

In [19]:
# Importaciones de librerías necesarias

import hashlib             # data_hash: huella de los datos
import os                  # leer variables de entorno
import time                # medir el tiempo de ejecución
from pathlib import Path   # rutas que funcionan en Windows y Linux

# --- Librerías de datos y gráficos ---
import matplotlib
matplotlib.use("Agg")      # backend sin ventana: las figuras se guardan en MLflow, no se muestran
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# --- MLflow ---
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature

# --- scikit-learn ---
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, ndcg_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

print("Librerías importadas correctamente")

Librerías importadas correctamente


In [20]:
# --- Configuración ---
T0 = time.time()  # inicio del cronómetro (criterio: correr en menos de ~3 minutos)

# Si existe un archivo .env junto al notebook, carga sus variables sin pisar las que
# ya estén definidas en el entorno. Así no depende de la terminal ni del editor.
ENV_FILE = Path(".env")
if ENV_FILE.exists():
    for line in ENV_FILE.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, value = line.split("=", 1)
            os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))
    print("Variables cargadas desde", ENV_FILE.resolve())

# Texto falso a propósito: si la sección 1 lo detecta, falta definir el URI real
PLACEHOLDER_URI = "REEMPLAZAR-CON-EL-URI-DEL-SERVIDOR"

# Lee la variable de entorno; si no existe, usa el placeholder
TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", PLACEHOLDER_URI)

EXPERIMENT_NAME = "course-recommender-demo"
PARENT_RUN_NAME = "model-comparison"
SEED = 42                # una sola semilla para todo el proyecto
K = 5                    # cuántos cursos se recomiendan (top-K)
TEST_SIZE = 0.2          # 80/20 por estudiante
PRIMARY_METRIC = f"ndcg_at_{K}"  # métrica que decide el ganador
DATA_DIR = Path("data")

print("URI leído:", TRACKING_URI)

Variables cargadas desde C:\Projects\mlflow-tracking-demo\.env
URI leído: sqlite:///mlflow.db


## 1. Conexión a MLflow

Esta celda comprueba que el notebook puede hablar con el servidor de tracking **antes** de hacer cualquier otra cosa. Si el URI sigue siendo el texto falso (porque nadie definió `MLFLOW_TRACKING_URI`) o si el servidor no responde, el notebook se detiene aquí con un mensaje claro. Es mejor fallar en este punto que varias celdas después con un error difícil de entender.

Si la conexión funciona, se crea (o se reutiliza) el experimento `course-recommender-demo`.

In [21]:
# 1. ¿Sigue siendo el placeholder? Entonces falta configurar el servidor
if TRACKING_URI == PLACEHOLDER_URI:
    raise RuntimeError(
        "MLFLOW_TRACKING_URI no está definida (sigue el placeholder). "
        "Defínela como variable de entorno o en un archivo .env junto al notebook "
        "(MLFLOW_TRACKING_URI=...) y reinicia el kernel. "
        "Para una prueba local usa: sqlite:///mlflow.db"
    )

# 2. Conectar
mlflow.set_tracking_uri(TRACKING_URI)
print("MLflow versión:", mlflow.__version__)

# 3. Comprobar que el servidor responde con una consulta de solo lectura
try:
    mlflow.search_experiments(max_results=1)
except Exception as e:
    raise RuntimeError(f"No se pudo conectar a MLflow en {TRACKING_URI}: {e}") from e

print("Conexión OK:", mlflow.get_tracking_uri())

# 4. Crear o reutilizar el experimento
experiment = mlflow.set_experiment(EXPERIMENT_NAME)
print("Experimento:", experiment.name, "| id:", experiment.experiment_id)
print("Artefactos en:", experiment.artifact_location)

MLflow versión: 3.17.2
Conexión OK: sqlite:///mlflow.db
Experimento: course-recommender-demo | id: 1
Artefactos en: file:c:/Projects/mlflow-tracking-demo/mlruns/1


## 2. Carga y validación de datos

Los tres CSV de `data/` son la **fuente de verdad** (los produce `scripts/generate_data.py` con semilla fija). Aquí se cargan y se validan con `assert`: si algo no cuadra, el notebook se detiene antes de entrenar nada.

| Tabla | Filas | Contenido |
|---|---|---|
| `estudiantes` | 2,000 | Perfil conocido el día que llega el estudiante |
| `cursos` | 60 | Catálogo |
| `eventos` | 40,000 | 20 recomendaciones mostradas por estudiante y si completó el curso en 30 días (`completo_30d`) |

Al final se calcula `DATA_HASH`, una huella de los datos cargados que se guarda como tag en cada run (linaje: qué datos produjeron qué modelo).

In [22]:
# Fail-fast si falta la carpeta de datos (p. ej., si se copió solo el .ipynb)
CSV_FILES = {name: DATA_DIR / f"{name}.csv" for name in ("estudiantes", "cursos", "eventos")}
missing = [str(p) for p in CSV_FILES.values() if not p.exists()]
if missing:
    raise FileNotFoundError(
        f"Faltan archivos de datos: {missing}. "
        "Abre el notebook desde la raíz del repo (junto a la carpeta data/)."
    )

estudiantes = pd.read_csv(CSV_FILES["estudiantes"])
cursos = pd.read_csv(CSV_FILES["cursos"])
eventos = pd.read_csv(CSV_FILES["eventos"])

for name, frame in [("estudiantes", estudiantes), ("cursos", cursos), ("eventos", eventos)]:
    print(f"{name:<12} {frame.shape}")

estudiantes  (2000, 6)
cursos       (60, 5)
eventos      (40000, 3)


In [23]:
# Vocabulario compartido entre estudiantes y cursos
AREAS = {"analisis_datos", "machine_learning", "sql_bases_datos", "python", "visualizacion"}
NIVELES = ["principiante", "intermedio", "avanzado"]  # orden ordinal
OBJETIVOS = {"empleo", "mejorar_trabajo", "certificacion", "proyecto_personal"}
PLANES = {"gratis", "premium"}
N_SHOWN = 20  # cursos mostrados por estudiante

# Columnas esperadas
assert list(estudiantes.columns) == ["student_id", "horas_semana", "objetivo_aprendizaje",
                                     "nivel_experiencia", "area_interes", "plan"], estudiantes.columns
assert list(cursos.columns) == ["course_id", "titulo", "categoria", "nivel", "duracion_horas"], cursos.columns
assert list(eventos.columns) == ["student_id", "course_id", "completo_30d"], eventos.columns

# Tamaños
assert len(estudiantes) == 2_000, len(estudiantes)
assert len(cursos) == 60, len(cursos)
assert len(eventos) == 40_000, len(eventos)

# Sin nulos
for name, frame in [("estudiantes", estudiantes), ("cursos", cursos), ("eventos", eventos)]:
    assert frame.notna().all().all(), f"Hay nulos en {name}"

# Llaves únicas e integridad referencial
assert estudiantes["student_id"].is_unique, "student_id duplicado"
assert cursos["course_id"].is_unique, "course_id duplicado"
assert not eventos.duplicated(["student_id", "course_id"]).any(), "pares (student_id, course_id) duplicados"
assert eventos["student_id"].isin(estudiantes["student_id"]).all(), "eventos con student_id desconocido"
assert eventos["course_id"].isin(cursos["course_id"]).all(), "eventos con course_id desconocido"

# Exactamente 20 recomendaciones por estudiante
assert (eventos.groupby("student_id").size() == N_SHOWN).all(), "no todos los estudiantes tienen 20 filas"
assert eventos["student_id"].nunique() == len(estudiantes)

# Vocabulario
assert set(estudiantes["area_interes"]) <= AREAS
assert set(cursos["categoria"]) <= AREAS
assert set(estudiantes["nivel_experiencia"]) <= set(NIVELES)
assert set(cursos["nivel"]) <= set(NIVELES)
assert set(estudiantes["objetivo_aprendizaje"]) <= OBJETIVOS
assert set(estudiantes["plan"]) <= PLANES

# Rangos numéricos
assert estudiantes["horas_semana"].between(1, 15).all()
assert cursos["duracion_horas"].between(2, 20).all()

# Target binario y tasa global entre 25% y 35%
assert set(eventos["completo_30d"].unique()) <= {0, 1}
tasa = eventos["completo_30d"].mean()
assert 0.25 <= tasa <= 0.35, f"Tasa de completación fuera de rango: {tasa:.3f}"

print(f"Todas las validaciones pasaron. Tasa de completación: {tasa:.1%}")

Todas las validaciones pasaron. Tasa de completación: 30.1%


In [24]:
# Huella de los datos cargados (linaje). No se compara contra un valor esperado.
def frame_bytes(frame: pd.DataFrame) -> bytes:
    """Bytes estables de un DataFrame (contenido + nombres de columnas)."""
    return ",".join(frame.columns).encode() + pd.util.hash_pandas_object(frame, index=False).to_numpy().tobytes()

hasher = hashlib.sha256()
for frame in (estudiantes, cursos, eventos):
    hasher.update(frame_bytes(frame))
DATA_HASH = hasher.hexdigest()[:16]

print("DATA_HASH:", DATA_HASH)

DATA_HASH: 2a83ad6e974b6097


## 3. Preparación: unión, variables de cruce y división

1. **Unión.** Cada fila de `eventos` (un par estudiante-curso mostrado) se completa con el perfil del estudiante y los atributos del curso.
2. **Variables de cruce.** Relacionan al estudiante con el curso:
   - `area_coincide`: 1 si la categoría del curso coincide con el área de interés del estudiante.
   - `brecha_nivel`: nivel del curso − nivel del estudiante (principiante=0, intermedio=1, avanzado=2), de −2 a +2.
3. **Variables de entrada.** Los tres modelos reciben el **mismo DataFrame crudo**; cada `Pipeline` elige sus columnas. Popularidad usa solo `course_id`; regresión logística y boosting usan el perfil, el curso y las variables de cruce. La variable `carga` (`duracion_horas / horas_semana`) participa en el generador de datos pero **no se entrega** a ningún modelo: el boosting debería poder aproximarla por su cuenta.
4. **División por estudiante** (`GroupShuffleSplit`, 80/20): todas las filas de un estudiante quedan del mismo lado. Las métricas de ranking necesitan las 20 filas completas de cada estudiante de prueba. No hay conjunto de validación porque no se ajustan hiperparámetros.

`add_features` es una función porque la sección 6 la reutiliza para puntuar el catálogo completo.

In [25]:
NIVEL_ORDINAL = {nivel: i for i, nivel in enumerate(NIVELES)}

ID_FEATURES = ["course_id"]  # solo lo usa el modelo de popularidad
CAT_FEATURES = ["objetivo_aprendizaje", "nivel_experiencia", "area_interes", "plan", "categoria", "nivel"]
NUM_FEATURES = ["horas_semana", "duracion_horas", "area_coincide", "brecha_nivel"]
INPUT_COLUMNS = ID_FEATURES + CAT_FEATURES + NUM_FEATURES
TARGET = "completo_30d"

assert "carga" not in INPUT_COLUMNS  # decisión de diseño: la carga no se entrega


def add_features(pairs: pd.DataFrame) -> pd.DataFrame:
    """Une pares (student_id, course_id) con sus perfiles y crea las variables de cruce."""
    out = (
        pairs
        .merge(estudiantes, on="student_id", how="left", validate="many_to_one")
        .merge(cursos, on="course_id", how="left", validate="many_to_one")
    )
    out["area_coincide"] = (out["categoria"] == out["area_interes"]).astype(int)
    out["brecha_nivel"] = out["nivel"].map(NIVEL_ORDINAL) - out["nivel_experiencia"].map(NIVEL_ORDINAL)
    # Numéricas como float: evita avisos de MLflow sobre enteros sin valores faltantes en la signature
    out[NUM_FEATURES] = out[NUM_FEATURES].astype(float)
    return out


df = add_features(eventos)
assert len(df) == len(eventos)
assert df[INPUT_COLUMNS].notna().all().all()

df[["student_id", "course_id"] + CAT_FEATURES + NUM_FEATURES + [TARGET]].head()

,student_id,course_id,objetivo_aprendizaje,nivel_experiencia,area_interes,plan,categoria,nivel,horas_semana,duracion_horas,area_coincide,brecha_nivel,completo_30d
0,E0001,C003,empleo,principiante,machine_learning,gratis,analisis_datos,principiante,4.0,6.0,0.0,0.0,1
1,E0001,C007,empleo,principiante,machine_learning,gratis,analisis_datos,intermedio,4.0,14.0,0.0,1.0,0
2,E0001,C008,empleo,principiante,machine_learning,gratis,analisis_datos,intermedio,4.0,14.0,0.0,1.0,1
3,E0001,C014,empleo,principiante,machine_learning,gratis,machine_learning,principiante,4.0,2.0,1.0,0.0,0
4,E0001,C017,empleo,principiante,machine_learning,gratis,machine_learning,principiante,4.0,7.0,1.0,0.0,1


In [26]:
splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=SEED)
train_idx, test_idx = next(splitter.split(df, groups=df["student_id"]))

# Orden fijo (estudiante, curso): permite reorganizar la prueba como matriz estudiantes x 20
train_df = df.iloc[train_idx].sort_values(["student_id", "course_id"]).reset_index(drop=True)
test_df = df.iloc[test_idx].sort_values(["student_id", "course_id"]).reset_index(drop=True)

X_train, y_train = train_df[INPUT_COLUMNS], train_df[TARGET]
X_test, y_test = test_df[INPUT_COLUMNS], test_df[TARGET]

# Ningún estudiante en ambos lados, y cada estudiante de prueba con sus 20 filas
assert set(train_df["student_id"]).isdisjoint(test_df["student_id"])
assert (test_df.groupby("student_id").size() == N_SHOWN).all()

N_TRAIN_STUDENTS = train_df["student_id"].nunique()
N_TEST_STUDENTS = test_df["student_id"].nunique()

print(f"Entrenamiento: {N_TRAIN_STUDENTS} estudiantes, {len(train_df):,} filas, tasa {y_train.mean():.1%}")
print(f"Prueba:        {N_TEST_STUDENTS} estudiantes, {len(test_df):,} filas, tasa {y_test.mean():.1%}")

Entrenamiento: 1600 estudiantes, 32,000 filas, tasa 30.1%
Prueba:        400 estudiantes, 8,000 filas, tasa 30.3%


## 4. Entrenamiento y registro en MLflow

### 4.1 Modelos

Cada modelo es un `Pipeline` de scikit-learn con el preprocesamiento incluido, así el modelo guardado en MLflow recibe **datos crudos**.

| Run | Modelo | Variables | Rol |
|---|---|---|---|
| `popularity` | Regresión logística sobre `course_id` (one-hot) | Solo `course_id` | Línea base: equivale a la tasa de completación por curso |
| `logreg` | Regresión logística | Perfil + curso + cruces | Modelo simple e interpretable |
| `hgb` | `HistGradientBoostingClassifier` | Las mismas que `logreg` | Capta no linealidades (como la carga) |

Los hiperparámetros están fijos en `MODEL_CONFIGS` (no se ajustan) y se registran como parámetros de cada run.

In [27]:
MODEL_CONFIGS = {
    "popularity": {"family": "baseline_popularity", "params": {"C": 1.0, "max_iter": 1000}},
    "logreg": {"family": "linear", "params": {"C": 1.0, "max_iter": 1000}},
    "hgb": {
        "family": "gradient_boosting",
        "params": {
            "learning_rate": 0.05,
            "max_iter": 300,
            "max_leaf_nodes": 15,
            "min_samples_leaf": 40,
            "l2_regularization": 1.0,
            "early_stopping": False,  # sin validación interna: misma división para todos
        },
    },
}

# MLflow 3.x serializa los modelos sklearn con skops (más seguro que pickle) y exige
# declarar los tipos internos que no reconoce. Solo el boosting los necesita; son
# objetos propios de scikit-learn creados por este mismo notebook.
SKOPS_TRUSTED_TYPES = {
    "hgb": [
        "functools.partial",
        "sklearn.ensemble._hist_gradient_boosting.predictor.TreePredictor",
        "sklearn.utils.validation.check_array",
    ],
}


def build_pipeline(name: str) -> Pipeline:
    """Devuelve un Pipeline nuevo (sin entrenar) para el modelo indicado."""
    params = MODEL_CONFIGS[name]["params"]

    if name == "popularity":
        preprocess = ColumnTransformer(
            [("course", OneHotEncoder(handle_unknown="ignore"), ID_FEATURES)]
        )
        clf = LogisticRegression(**params, random_state=SEED)

    elif name == "logreg":
        preprocess = ColumnTransformer([
            ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_FEATURES),
            ("num", StandardScaler(), NUM_FEATURES),
        ])
        clf = LogisticRegression(**params, random_state=SEED)

    elif name == "hgb":
        preprocess = ColumnTransformer([
            ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), CAT_FEATURES),
            ("num", "passthrough", NUM_FEATURES),
        ])
        # Las primeras columnas de salida son las categóricas (mismo orden del ColumnTransformer)
        is_categorical = [True] * len(CAT_FEATURES) + [False] * len(NUM_FEATURES)
        clf = HistGradientBoostingClassifier(**params, categorical_features=is_categorical, random_state=SEED)

    else:
        raise ValueError(f"Modelo desconocido: {name}")

    return Pipeline([("preprocess", preprocess), ("model", clf)])


for name in MODEL_CONFIGS:
    print(name, "->", build_pipeline(name).named_steps["model"].__class__.__name__)

popularity -> LogisticRegression
logreg -> LogisticRegression
hgb -> HistGradientBoostingClassifier


### 4.2 Métricas

La evaluación es **offline**: solo conocemos el resultado de los 20 cursos que se le mostraron a cada estudiante. "Recomendar" significa ordenar esos 20 cursos por probabilidad predicha y quedarse con los `K=5` primeros.

| Métrica | Qué mide | Nivel |
|---|---|---|
| `roc_auc` | Si el modelo ordena bien los pares completados por encima de los no completados | Todos los pares mezclados |
| `log_loss` | Qué tan acertadas son las probabilidades (menor es mejor) | Todos los pares |
| `precision_at_5` | De los 5 recomendados, qué fracción se completó | Por estudiante, promediado |
| `ndcg_at_5` | Igual, pero premia que los aciertos estén más arriba. **Primaria** | Por estudiante, promediado |
| `coverage_at_5` | Fracción del catálogo (60 cursos) que aparece en algún top-5 | Todo el conjunto de prueba |

In [28]:
def ranking_matrices(frame: pd.DataFrame, scores: np.ndarray):
    """Reorganiza (estudiante, curso) ordenados en matrices n_estudiantes x 20."""
    n_students = frame["student_id"].nunique()
    shape = (n_students, N_SHOWN)
    y = frame[TARGET].to_numpy().reshape(shape)
    s = np.asarray(scores).reshape(shape)
    c = frame["course_id"].to_numpy().reshape(shape)
    return y, s, c


def evaluate(frame: pd.DataFrame, proba: np.ndarray, k: int = K) -> dict:
    """Calcula las 5 métricas sobre un conjunto ordenado por (student_id, course_id)."""
    y, s, c = ranking_matrices(frame, proba)
    top_k = np.argsort(-s, axis=1, kind="stable")[:, :k]  # índices de los k mejores por fila
    rows = np.arange(len(y))[:, None]
    return {
        "roc_auc": roc_auc_score(frame[TARGET], proba),
        "log_loss": log_loss(frame[TARGET], proba),
        f"precision_at_{k}": float(y[rows, top_k].mean()),
        f"ndcg_at_{k}": float(ndcg_score(y, s, k=k)),
        f"coverage_at_{k}": len(np.unique(c[rows, top_k])) / len(cursos),
    }


def top_k_sample(frame: pd.DataFrame, proba: np.ndarray, n_students: int = 5, k: int = K) -> pd.DataFrame:
    """Top-k legible (con títulos) para los primeros n estudiantes de prueba."""
    sample_ids = frame["student_id"].drop_duplicates().iloc[:n_students]
    out = frame.assign(score=proba)[frame["student_id"].isin(sample_ids)]
    out = out.sort_values(["student_id", "score"], ascending=[True, False]).groupby("student_id").head(k)
    out["rank"] = out.groupby("student_id").cumcount() + 1
    return out[["student_id", "rank", "course_id", "titulo", "categoria", "nivel",
                "area_interes", "nivel_experiencia", "score", TARGET]].reset_index(drop=True)


def importance_figure(pipe: Pipeline, X: pd.DataFrame, y: pd.Series, title: str):
    """Importancia por permutación (caída de ROC AUC) sobre las variables crudas."""
    result = permutation_importance(pipe, X, y, scoring="roc_auc", n_repeats=5, random_state=SEED)
    imp = (
        pd.DataFrame({"feature": X.columns, "mean": result.importances_mean, "std": result.importances_std})
        .query("feature not in @ID_FEATURES")  # course_id no lo usan logreg ni hgb
        .sort_values("mean")
    )
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.barh(imp["feature"], imp["mean"], xerr=imp["std"], color="#4C72B0")
    ax.set_xlabel("Caída de ROC AUC al permutar la variable")
    ax.set_title(title)
    fig.tight_layout()
    return fig

### 4.3 Registro manual en MLflow

`train_and_log` entrena un modelo dentro de su propio run y registra **manualmente** (sin `autolog`, para que el DE vea exactamente qué se guarda):

| Qué | Detalle |
|---|---|
| Parámetros | `seed`, `test_size`, `k`, `n_train_estudiantes`, `n_test_estudiantes`, variables usadas e hiperparámetros (`model__*`) |
| Métricas | Las 5 de la tabla anterior |
| Tags | `model_family`, `data_hash`, `caso=ficticio` |
| Artefactos | `top5_muestra.csv` (todos); `importancia_variables.png` (solo `logreg` y `hgb`) |
| Modelo | `model`, con `signature` e `input_example` (datos crudos) |

In [29]:
COMMON_PARAMS = {
    "seed": SEED,
    "test_size": TEST_SIZE,
    "k": K,
    "n_train_estudiantes": N_TRAIN_STUDENTS,
    "n_test_estudiantes": N_TEST_STUDENTS,
}
COMMON_TAGS = {"data_hash": DATA_HASH, "caso": "ficticio"}


def features_used(name: str) -> list:
    return ID_FEATURES if name == "popularity" else CAT_FEATURES + NUM_FEATURES


def train_and_log(name: str, run_name: str | None = None, nested: bool = True, extra_tags: dict | None = None) -> dict:
    """Entrena un modelo, lo evalúa y registra todo en un run de MLflow."""
    cfg = MODEL_CONFIGS[name]
    pipe = build_pipeline(name)

    with mlflow.start_run(run_name=run_name or name, nested=nested) as run:
        # Entrenamiento y evaluación
        pipe.fit(X_train, y_train)
        proba = pipe.predict_proba(X_test)[:, 1]
        metrics = evaluate(test_df, proba)

        # Parámetros, métricas y tags
        mlflow.log_params({
            **COMMON_PARAMS,
            "model_name": name,
            "features": ",".join(features_used(name)),
            **{f"model__{p}": v for p, v in cfg["params"].items()},
        })
        mlflow.log_metrics(metrics)
        mlflow.set_tags({**COMMON_TAGS, "model_family": cfg["family"], **(extra_tags or {})})

        # Artefactos
        mlflow.log_text(top_k_sample(test_df, proba).to_csv(index=False), "top5_muestra.csv")
        if name != "popularity":
            fig = importance_figure(pipe, X_test, y_test, f"Importancia de variables: {name}")
            mlflow.log_figure(fig, "importancia_variables.png")
            plt.close(fig)

        # Modelo con signature (entrada cruda -> clase predicha) e input_example
        sample = X_train.head(5)
        signature = infer_signature(sample, pipe.predict(sample))
        mlflow.sklearn.log_model(
            sk_model=pipe,
            name="model",
            signature=signature,
            input_example=sample,
            skops_trusted_types=SKOPS_TRUSTED_TYPES.get(name),
        )

    return {"name": name, "run_id": run.info.run_id, "metrics": metrics, "proba": proba}

### 4.4 Corrida: run padre y tres hijos

Un run padre `model-comparison` agrupa los tres runs hijos. El padre registra los parámetros comunes; cada hijo, lo suyo. Esta celda es la que más tarda (decenas de segundos, según el servidor).

In [30]:
results = {}

with mlflow.start_run(run_name=PARENT_RUN_NAME) as parent_run:
    mlflow.log_params(COMMON_PARAMS)
    mlflow.set_tags({**COMMON_TAGS, "run_role": "parent"})

    for name in MODEL_CONFIGS:
        t = time.time()
        results[name] = train_and_log(name)
        print(f"{name:<11} run_id={results[name]['run_id']}  "
              f"{PRIMARY_METRIC}={results[name]['metrics'][PRIMARY_METRIC]:.4f}  ({time.time() - t:.1f} s)")

PARENT_RUN_ID = parent_run.info.run_id
print("\nRun padre:", PARENT_RUN_ID)

comparison = pd.DataFrame({name: r["metrics"] for name, r in results.items()}).T
comparison.round(4)

popularity  run_id=2423b735584742e78f9c6cb4725d618a  ndcg_at_5=0.4521  (7.0 s)
logreg      run_id=3bea8d91666e40f583050997737c3306  ndcg_at_5=0.5556  (6.3 s)
hgb         run_id=f6995850794a4263a9ebba7eae10f5e4  ndcg_at_5=0.5768  (10.5 s)

Run padre: 2dd430dccf2d4bf39b696f693461161e


,roc_auc,log_loss,precision_at_5,ndcg_at_5,coverage_at_5
popularity,0.6415,0.5846,0.4110,0.4521,0.4667
logreg,0.7220,0.5462,0.4805,0.5556,0.8000
hgb,0.7364,0.5343,0.5015,0.5768,1.0000


In [31]:
# Hipótesis del diseño (se informan, no detienen el notebook)
ndcg = comparison[PRIMARY_METRIC]
auc = comparison["roc_auc"]

order_ok = ndcg["popularity"] < ndcg["logreg"] <= ndcg["hgb"]
print(f"[Hipótesis] popularidad < logreg <= hgb en {PRIMARY_METRIC}: {'se cumple' if order_ok else 'NO se cumple'}")

same_order = list(ndcg.sort_values().index) == list(auc.sort_values().index)
print(f"[Hipótesis] {PRIMARY_METRIC} y roc_auc ordenan igual los modelos: {'sí' if same_order else 'NO (hallazgo a revisar)'}")

gap = ndcg["hgb"] - ndcg["logreg"]
print(f"Ventaja de hgb sobre logreg en {PRIMARY_METRIC}: {gap:+.4f}")

[Hipótesis] popularidad < logreg <= hgb en ndcg_at_5: se cumple
[Hipótesis] ndcg_at_5 y roc_auc ordenan igual los modelos: sí
Ventaja de hgb sobre logreg en ndcg_at_5: +0.0212


## 5. Selección del mejor modelo con `search_runs`

El ganador no se toma de la memoria del notebook: se **consulta al servidor de tracking** con `mlflow.search_runs`, filtrando los hijos del run padre y ordenando por `ndcg_at_5`. Así se demuestra que lo registrado es suficiente para decidir.

Después se reabre el run padre para anotar `best_ndcg_at_5` y el nombre del ganador.

In [32]:
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string=f"tags.mlflow.parentRunId = '{PARENT_RUN_ID}'",
    order_by=[f"metrics.{PRIMARY_METRIC} DESC"],
)
assert len(runs) == len(MODEL_CONFIGS), f"Se esperaban {len(MODEL_CONFIGS)} runs hijos, hay {len(runs)}"

metric_cols = [f"metrics.{m}" for m in results["popularity"]["metrics"]]
display(runs[["tags.mlflow.runName", "run_id"] + metric_cols].round(4))

best = runs.iloc[0]
BEST_RUN_ID = best["run_id"]
BEST_MODEL_NAME = best["tags.mlflow.runName"]
BEST_SCORE = best[f"metrics.{PRIMARY_METRIC}"]

# Anotar el resultado en el run padre
with mlflow.start_run(run_id=PARENT_RUN_ID):
    mlflow.log_metric(f"best_{PRIMARY_METRIC}", BEST_SCORE)
    mlflow.set_tags({"best_model": BEST_MODEL_NAME, "best_run_id": BEST_RUN_ID})

print(f"Ganador: {BEST_MODEL_NAME} (run {BEST_RUN_ID}) con {PRIMARY_METRIC} = {BEST_SCORE:.4f}")

,tags.mlflow.runName,run_id,metrics.roc_auc,metrics.log_loss,metrics.precision_at_5,metrics.ndcg_at_5,metrics.coverage_at_5
0,hgb,f6995850794a4263a9ebba7eae10f5e4,0.7364,0.5343,0.5015,0.5768,1.0000
1,logreg,3bea8d91666e40f583050997737c3306,0.7220,0.5462,0.4805,0.5556,0.8000
2,popularity,2423b735584742e78f9c6cb4725d618a,0.6415,0.5846,0.4110,0.4521,0.4667


Ganador: hgb (run f6995850794a4263a9ebba7eae10f5e4) con ndcg_at_5 = 0.5768


## 6. Recomendaciones con el modelo ganador

Se carga el modelo **desde MLflow** (`runs:/<run_id>/model`), no desde la memoria del notebook. Primero se comprueba que el modelo cargado predice exactamente lo mismo que el entrenado. Luego se generan recomendaciones top-5 para 3 estudiantes de prueba.

Aquí sí se puntúan los **60 cursos del catálogo**, como se haría en producción (en la evaluación solo se podían usar los 20 mostrados, porque de los demás no conocemos el resultado). La columna `mostrado` indica si el curso estaba entre los 20 que se le mostraron al estudiante.

In [33]:
model_uri = f"runs:/{BEST_RUN_ID}/model"
best_model = mlflow.sklearn.load_model(model_uri)
print("Modelo cargado desde:", model_uri)

# El modelo cargado debe reproducir las predicciones del entrenado
np.testing.assert_allclose(
    best_model.predict_proba(X_test)[:, 1], results[BEST_MODEL_NAME]["proba"], rtol=0, atol=1e-12
)
print("El modelo cargado reproduce las predicciones del run.")

Modelo cargado desde: runs:/f6995850794a4263a9ebba7eae10f5e4/model
El modelo cargado reproduce las predicciones del run.


In [34]:
demo_students = test_df["student_id"].drop_duplicates().iloc[:3].tolist()

# Todos los pares (estudiante, curso) del catálogo para esos 3 estudiantes
candidates = add_features(
    pd.MultiIndex.from_product([demo_students, cursos["course_id"]], names=["student_id", "course_id"])
    .to_frame(index=False)
)
candidates["score"] = best_model.predict_proba(candidates[INPUT_COLUMNS])[:, 1]

shown = set(zip(eventos["student_id"], eventos["course_id"]))
candidates["mostrado"] = [(s, c) in shown for s, c in zip(candidates["student_id"], candidates["course_id"])]

recommendations = (
    candidates.sort_values(["student_id", "score"], ascending=[True, False])
    .groupby("student_id").head(K)
    .assign(rank=lambda d: d.groupby("student_id").cumcount() + 1)
)

for sid, recs in recommendations.groupby("student_id"):
    p = estudiantes.set_index("student_id").loc[sid]
    print(f"\n{sid}: área={p.area_interes}, nivel={p.nivel_experiencia}, "
          f"horas/semana={p.horas_semana}, objetivo={p.objetivo_aprendizaje}, plan={p.plan}")
    display(recs[["rank", "course_id", "titulo", "categoria", "nivel", "duracion_horas", "score", "mostrado"]]
            .round({"score": 3}).reset_index(drop=True))


E0024: área=analisis_datos, nivel=avanzado, horas/semana=4, objetivo=empleo, plan=gratis


,rank,course_id,titulo,categoria,nivel,duracion_horas,score,mostrado
0,1,C012,Análisis de datos avanzado 3,analisis_datos,avanzado,10.0,0.612,False
1,2,C002,Análisis de datos básico 2,analisis_datos,principiante,5.0,0.593,True
2,3,C005,Análisis de datos básico 5,analisis_datos,principiante,4.0,0.590,True
3,4,C009,Análisis de datos intermedio 4,analisis_datos,intermedio,6.0,0.569,False
4,5,C006,Análisis de datos intermedio 1,analisis_datos,intermedio,9.0,0.562,False



E0030: área=python, nivel=principiante, horas/semana=5, objetivo=mejorar_trabajo, plan=premium


,rank,course_id,titulo,categoria,nivel,duracion_horas,score,mostrado
0,1,C039,Python básico 3,python,principiante,3.0,0.720,False
1,2,C038,Python básico 2,python,principiante,2.0,0.713,False
2,3,C037,Python básico 1,python,principiante,5.0,0.685,True
3,4,C040,Python básico 4,python,principiante,6.0,0.668,True
4,5,C041,Python básico 5,python,principiante,6.0,0.668,False



E0031: área=machine_learning, nivel=principiante, horas/semana=11, objetivo=mejorar_trabajo, plan=gratis


,rank,course_id,titulo,categoria,nivel,duracion_horas,score,mostrado
0,1,C014,Machine learning básico 2,machine_learning,principiante,2.0,0.862,False
1,2,C015,Machine learning básico 3,machine_learning,principiante,4.0,0.776,True
2,3,C013,Machine learning básico 1,machine_learning,principiante,6.0,0.763,True
3,4,C016,Machine learning básico 4,machine_learning,principiante,6.0,0.763,False
4,5,C017,Machine learning básico 5,machine_learning,principiante,7.0,0.732,True


## 7. Reproducibilidad

Se reentrena el modelo ganador desde cero, con la misma semilla y los mismos datos, en un run aparte (`reproducibility-check-<modelo>`, tag `reproducibility_check=true`). Luego se comparan sus métricas con las que quedaron **registradas en el servidor** para el run original. Si difieren más allá de una tolerancia numérica mínima, el `assert` falla.

Este run queda fuera del run padre a propósito: no compite en la comparación.

In [35]:
repro = train_and_log(
    BEST_MODEL_NAME,
    run_name=f"reproducibility-check-{BEST_MODEL_NAME}",
    nested=False,
    extra_tags={"reproducibility_check": "true", "reproduces_run_id": BEST_RUN_ID},
)

TOLERANCE = 1e-9
check = pd.DataFrame({
    "original (servidor)": {m: best[f"metrics.{m}"] for m in repro["metrics"]},
    "reentrenado": repro["metrics"],
})
check["diferencia"] = (check["reentrenado"] - check["original (servidor)"]).abs()
display(check)

for metric, row in check.iterrows():
    assert row["diferencia"] <= TOLERANCE, f"{metric} no se reproduce: diferencia {row['diferencia']:.2e}"

print(f"Reproducibilidad OK (tolerancia {TOLERANCE}). Run de verificación: {repro['run_id']}")

,original (servidor),reentrenado,diferencia
roc_auc,0.736418,0.736418,0.0
log_loss,0.534305,0.534305,0.0
precision_at_5,0.501500,0.501500,0.0
ndcg_at_5,0.576772,0.576772,0.0
coverage_at_5,1.000000,1.000000,0.0


Reproducibilidad OK (tolerancia 1e-09). Run de verificación: 6284ddbf68a0439a99ad2e8fa4e21ff6


## Resumen

Al terminar, en la UI de MLflow debería verse:

- Experimento `course-recommender-demo` con el run padre `model-comparison` y sus tres hijos (`popularity`, `logreg`, `hgb`), cada uno con 5 métricas, parámetros, tags, artefactos y modelo con signature.
- En el run padre: `best_ndcg_at_5` y los tags `best_model` y `best_run_id`.
- Un run `reproducibility-check-<modelo>` con tag `reproducibility_check=true`.

In [36]:
elapsed = time.time() - T0
print(f"Tiempo total: {elapsed:.0f} s ({'<' if elapsed < 180 else '>='} 3 min)")
print(f"Experimento:  {EXPERIMENT_NAME} (id {experiment.experiment_id})")
print(f"Run padre:    {PARENT_RUN_ID}")
print(f"Ganador:      {BEST_MODEL_NAME} ({PRIMARY_METRIC} = {BEST_SCORE:.4f}), run {BEST_RUN_ID}")
print(f"Repro check:  {repro['run_id']}")

Tiempo total: 39 s (< 3 min)
Experimento:  course-recommender-demo (id 1)
Run padre:    2dd430dccf2d4bf39b696f693461161e
Ganador:      hgb (ndcg_at_5 = 0.5768), run f6995850794a4263a9ebba7eae10f5e4
Repro check:  6284ddbf68a0439a99ad2e8fa4e21ff6
